# 单机 1/4/8×RTX 4090：DDP/FSDP Benchmark

这个 Notebook 只保留服务器实验的高层操作：配置、预检、运行和展示。命令构造、断点复用、超时、日志、重复聚合、质量门禁、CSV/JSON/PNG 落盘都由 `distributed_bench_utils.py` 和 `run_dist_bench.py` 完成。

可以顺序执行全部单元。已完成且 identity 匹配的 raw case 会自动复用；只有设置 `rerun_existing=True` 才会强制重跑。

## 0. 会话配置

8 卡服务器保持默认 `(1, 4, 8)`；4 卡服务器改成 `(1, 4)`。Notebook kernel 必须使用安装了当前项目、PyTorch CUDA、pandas 和 matplotlib 的同一个 Python 环境。

In [1]:
import sys
from importlib import import_module
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'tests' / 'distributed_bench_utils.py').is_file()
)
sys.path.insert(0, str(ROOT / 'tests'))

bench_utils = import_module('distributed_bench_utils')
BenchmarkSettings = bench_utils.BenchmarkSettings
DistributedBenchmark = bench_utils.DistributedBenchmark

settings = BenchmarkSettings(
    world_sizes=(1, 4),
    strategies=('single', 'ddp', 'fsdp'),
    model_config='configs/model_125m_moe.yaml',
    local_batch=4,
    capacity_batches=(1, 2, 4, 8, 16, 32, 64, 128),
    warmup_steps=10,
    measure_steps=30,
    repeats=3,
)
bench = DistributedBenchmark(
    root=ROOT,
    output='artifacts/distributed_benchmark/rtx4090_125m_moe',
    settings=settings,
)

## 1. 服务器预检与硬件清单

预检会验证当前 kernel 的 Python/CUDA、可见 GPU 数、`torch.distributed`、`nvidia-smi`、模型配置和全部拓扑 YAML。硬件清单另外保存 GPU、驱动、PCIe/NVLink 拓扑、Git commit 与 dirty 状态。

In [2]:
preflight = bench.preflight()
inventory = bench.inventory()
preflight, inventory

RUN /home/ubuntu/mini-train-sys/.venv/bin/python /home/ubuntu/mini-train-sys/scripts/run_dist_bench.py inventory --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/inventory


{
  "timestamp_utc": "2026-07-20T19:28:44.435776+00:00",
  "hostname": "ml-ubuntu22-04-desktop-v3-2-128gb-25m",
  "platform": "Linux-5.15.0-72-generic-x86_64-with-glibc2.35",
  "python": "3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]",
  "cpu_count": 44,
  "torch": "2.5.1+cu118",
  "cuda": "11.8",
  "nvidia_smi": "GPU 0: NVIDIA GeForce RTX 4090 (UUID: GPU-800e09d3-f879-3678-3a81-d0b5be4877f4)\nGPU 1: NVIDIA GeForce RTX 4090 (UUID: GPU-0caa5d5c-3a92-327e-6a71-f81c81521bca)\nGPU 2: NVIDIA GeForce RTX 4090 (UUID: GPU-c1d23de5-ac10-5852-c4f5-8fa33b1bd368)\nGPU 3: NVIDIA GeForce RTX 4090 (UUID: GPU-2baa5c33-c7d5-2ce5-7953-44529fffab63)",
  "gpu_inventory": "0, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:29:00.0\n1, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:2C:00.0\n2, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:2F:00.0\n3, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:32:00.0",
  "topology": "\u001bGPU0\tGPU1\tGPU2\tGPU3\tCPU Affinity

({'ok': True,
  'root': '/home/ubuntu/mini-train-sys',
  'python': '/home/ubuntu/mini-train-sys/.venv/bin/python',
  'torch': '2.5.1+cu118',
  'cuda': '11.8',
  'visible_gpus': 4,
  'requested_world_sizes': (1, 4),
  'strategies': ('single', 'ddp', 'fsdp'),
  'output': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe',
  'errors': [],
  'warnings': []},
 {'timestamp_utc': '2026-07-20T19:28:44.435776+00:00',
  'hostname': 'ml-ubuntu22-04-desktop-v3-2-128gb-25m',
  'platform': 'Linux-5.15.0-72-generic-x86_64-with-glibc2.35',
  'python': '3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]',
  'cpu_count': 44,
  'torch': '2.5.1+cu118',
  'cuda': '11.8',
  'nvidia_smi': 'GPU 0: NVIDIA GeForce RTX 4090 (UUID: GPU-800e09d3-f879-3678-3a81-d0b5be4877f4)\nGPU 1: NVIDIA GeForce RTX 4090 (UUID: GPU-0caa5d5c-3a92-327e-6a71-f81c81521bca)\nGPU 2: NVIDIA GeForce RTX 4090 (UUID: GPU-c1d23de5-ac10-5852-c4f5-8fa33b1bd368)\nGPU 3: NVIDIA GeForce RTX 4090 (UUID: GPU-2baa5c33-c7d5-2ce5

## 2. Weak scaling

固定每卡 local batch，比较 1→4→8 卡的吞吐、P50/P95 step 延迟、扩展效率、数据等待和显存。默认同时运行 DDP 与 FSDP。每个 case 独立 `torchrun`，10 次 warmup、30 次测量、3 次重复。

In [3]:
weak = bench.run_weak()
weak.show()

RUN /home/ubuntu/mini-train-sys/.venv/bin/python /home/ubuntu/mini-train-sys/scripts/run_dist_bench.py run --suite weak --strategies single ddp fsdp --world-sizes 1 4 --model-config configs/model_125m_moe.yaml --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak --case-timeout-seconds 1800 --local-batch 4 --warmup-steps 10 --measure-steps 30 --repeats 3


{
  "timestamp_utc": "2026-07-20T19:28:47.150311+00:00",
  "hostname": "ml-ubuntu22-04-desktop-v3-2-128gb-25m",
  "platform": "Linux-5.15.0-72-generic-x86_64-with-glibc2.35",
  "python": "3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]",
  "cpu_count": 44,
  "torch": "2.5.1+cu118",
  "cuda": "11.8",
  "nvidia_smi": "GPU 0: NVIDIA GeForce RTX 4090 (UUID: GPU-800e09d3-f879-3678-3a81-d0b5be4877f4)\nGPU 1: NVIDIA GeForce RTX 4090 (UUID: GPU-0caa5d5c-3a92-327e-6a71-f81c81521bca)\nGPU 2: NVIDIA GeForce RTX 4090 (UUID: GPU-c1d23de5-ac10-5852-c4f5-8fa33b1bd368)\nGPU 3: NVIDIA GeForce RTX 4090 (UUID: GPU-2baa5c33-c7d5-2ce5-7953-44529fffab63)",
  "gpu_inventory": "0, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:29:00.0\n1, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:2C:00.0\n2, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:2F:00.0\n3, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:32:00.0",
  "topology": "\u001bGPU0\tGPU1\tGPU2\tGPU3\tCPU Affinity

RUN /home/ubuntu/mini-train-sys/.venv/bin/python -m torch.distributed.run --standalone --nproc_per_node 1 scripts/run_dist_bench.py _worker --config configs/server/rtx4090_24gb/runs/single_1gpu.yaml --model-config /home/ubuntu/mini-train-sys/configs/model_125m_moe.yaml --batch-size 4 --warmup-steps 10 --measure-steps 30 --repeat 1 --case-identity 6032b8d05b8efb3cbf11b6bf3ecea1e76b9e4a078fcdae99e61f3110dcbf2299 --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/raw/weak_single_1gpu_b4_r1.json


RUN /home/ubuntu/mini-train-sys/.venv/bin/python -m torch.distributed.run --standalone --nproc_per_node 1 scripts/run_dist_bench.py _worker --config configs/server/rtx4090_24gb/runs/single_1gpu.yaml --model-config /home/ubuntu/mini-train-sys/configs/model_125m_moe.yaml --batch-size 4 --warmup-steps 10 --measure-steps 30 --repeat 2 --case-identity 732924e69ef52abf3b4f8b54fadffabc235331af39ba659eef6bb9857456bf3b --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/raw/weak_single_1gpu_b4_r2.json


RUN /home/ubuntu/mini-train-sys/.venv/bin/python -m torch.distributed.run --standalone --nproc_per_node 1 scripts/run_dist_bench.py _worker --config configs/server/rtx4090_24gb/runs/ddp_1gpu.yaml --model-config /home/ubuntu/mini-train-sys/configs/model_125m_moe.yaml --batch-size 4 --warmup-steps 10 --measure-steps 30 --repeat 0 --case-identity 56b618b4a96275b2b1cc290661c99088fa92b0e50dd29484b8cc9f0f7431f4f7 --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/raw/weak_ddp_1gpu_b4_r0.json


...

{'runs_succeeded': 15,
 'runs_failed': 0,
 'all_requested_repeats_completed': True,
 'weak_scaling_efficiency_ge_80pct': False,
 'data_stall_le_5pct': True,
 'memory_headroom_ge_10pct': True}

<Figure size 1300x900 with 4 Axes>

Saved artifacts


{'results': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/results_aggregated.csv',
 'failures': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/failures.csv',
 'metrics': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/quality_gates.json',
 'figure': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/weak_overview.png',
 'readme': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/README.md'}

{'results': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/results_aggregated.csv'),
 'failures': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/failures.csv'),
 'metrics': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/quality_gates.json'),
 'figure': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/weak_overview.png'),
 'readme': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/weak/presentation/README.md')}

## 3. Capacity sweep

逐个扫描 local batch。每个 case 位于独立子进程，OOM、超时和其他失败会写入 failure 表而不会污染后续 case；图表给出各策略/卡数的最大成功 local batch 和成功 case 的显存曲线。

In [4]:
capacity = bench.run_capacity()
capacity.show()

RUN /home/ubuntu/mini-train-sys/.venv/bin/python /home/ubuntu/mini-train-sys/scripts/run_dist_bench.py run --suite capacity --strategies single ddp fsdp --world-sizes 1 4 --model-config configs/model_125m_moe.yaml --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity --case-timeout-seconds 1800 --batch-sizes 1 2 4 8 16 32 64 128 --warmup-steps 3 --measure-steps 5 --repeats 1


{
  "timestamp_utc": "2026-07-20T19:34:45.295043+00:00",
  "hostname": "ml-ubuntu22-04-desktop-v3-2-128gb-25m",
  "platform": "Linux-5.15.0-72-generic-x86_64-with-glibc2.35",
  "python": "3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]",
  "cpu_count": 44,
  "torch": "2.5.1+cu118",
  "cuda": "11.8",
  "nvidia_smi": "GPU 0: NVIDIA GeForce RTX 4090 (UUID: GPU-800e09d3-f879-3678-3a81-d0b5be4877f4)\nGPU 1: NVIDIA GeForce RTX 4090 (UUID: GPU-0caa5d5c-3a92-327e-6a71-f81c81521bca)\nGPU 2: NVIDIA GeForce RTX 4090 (UUID: GPU-c1d23de5-ac10-5852-c4f5-8fa33b1bd368)\nGPU 3: NVIDIA GeForce RTX 4090 (UUID: GPU-2baa5c33-c7d5-2ce5-7953-44529fffab63)",
  "gpu_inventory": "0, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:29:00.0\n1, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:2C:00.0\n2, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:2F:00.0\n3, NVIDIA GeForce RTX 4090, 24564 MiB, 525.105.17, 00000000:32:00.0",
  "topology": "\u001bGPU0\tGPU1\tGPU2\tGPU3\tCPU Affinity

RUN /home/ubuntu/mini-train-sys/.venv/bin/python -m torch.distributed.run --standalone --nproc_per_node 1 scripts/run_dist_bench.py _worker --config configs/server/rtx4090_24gb/runs/single_1gpu.yaml --model-config /home/ubuntu/mini-train-sys/configs/model_125m_moe.yaml --batch-size 2 --warmup-steps 3 --measure-steps 5 --repeat 0 --case-identity 16129dfe1424b9a9c1f481812ee3b151c8f0af170b004e4849d4b76a3a624daf --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/raw/capacity_single_1gpu_b2_r0.json


RUN /home/ubuntu/mini-train-sys/.venv/bin/python -m torch.distributed.run --standalone --nproc_per_node 1 scripts/run_dist_bench.py _worker --config configs/server/rtx4090_24gb/runs/single_1gpu.yaml --model-config /home/ubuntu/mini-train-sys/configs/model_125m_moe.yaml --batch-size 4 --warmup-steps 3 --measure-steps 5 --repeat 0 --case-identity 44dcc4767c43cd5ba6e8b98fbc4fffeedbecd8441a41012a092d909af473191f --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/raw/capacity_single_1gpu_b4_r0.json


RUN /home/ubuntu/mini-train-sys/.venv/bin/python -m torch.distributed.run --standalone --nproc_per_node 1 scripts/run_dist_bench.py _worker --config configs/server/rtx4090_24gb/runs/single_1gpu.yaml --model-config /home/ubuntu/mini-train-sys/configs/model_125m_moe.yaml --batch-size 8 --warmup-steps 3 --measure-steps 5 --repeat 0 --case-identity 46ac1b58cfcf7188a938965fc61a01f87ed240b8587147bd4ff1ebf7b9ea08cd --output /data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/raw/capacity_single_1gpu_b8_r0.json


...

,strategy,world_size,batch_size,repeat,oom,timed_out,returncode,log,error_tail
0,single,1,64,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
1,single,1,128,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
2,ddp,1,64,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
3,ddp,1,128,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
4,ddp,4,64,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
5,ddp,4,128,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
6,fsdp,1,64,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
7,fsdp,1,128,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
8,fsdp,4,64,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,lastic/multiprocessing/api.py:869] failed (exi...
9,fsdp,4,128,0,True,False,1,/data/mini-train-sys/artifacts/distributed_ben...,"main\n return _run_code(code, main_globals,..."


<Figure size 1300x500 with 2 Axes>

Saved artifacts


{'results': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/results_aggregated.csv',
 'failures': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/failures.csv',
 'metrics': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/capacity_frontier.json',
 'figure': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/capacity_overview.png',
 'readme': '/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/README.md'}

{'results': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/results_aggregated.csv'),
 'failures': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/failures.csv'),
 'metrics': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/capacity_frontier.json'),
 'figure': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/capacity_overview.png'),
 'readme': PosixPath('/data/mini-train-sys/artifacts/distributed_benchmark/rtx4090_125m_moe/capacity/presentation/README.md')}

## 4. 只重新打开已有结果

重启 Notebook 后不需要重跑实验。使用下面两行重新加载 summary、表格和图片。若确实要无条件重跑 raw case，调用 `bench.run_weak(rerun_existing=True)` 或 `bench.run_capacity(rerun_existing=True)`。

In [5]:
# bench.load('weak').show()
# bench.load('capacity').show()